# SPF extraction and DataFrame loader

This notebook starts from a user-supplied `.spf` file, extracts the embedded files, and then loads whichever XML files are present into pandas DataFrames.

Behaviour:
- user sets `SPF_FILE`
- notebook extracts files from the SPF container
- if a target file is missing, it is skipped
- available XML files are parsed into DataFrames

This assumes the `.spf` file behaves like a ZIP/container, which matches your current files.


In [1]:
from pathlib import Path
import re
import shutil
import zipfile
import xml.etree.ElementTree as ET
from typing import Any, Dict, Iterable, List, Optional

import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)


## User inputs

Set the `.spf` file name or full path here.


In [ ]:
# File path and name
SPF_FILE = Path('../data/1505708.spf')

# Where extracted contents should go.
EXTRACT_ROOT = Path('extracted_spf')

# Files we care about. If they do not exist in the SPF, they will be skipped.
TARGET_FILES = [
    'Header.Xml',
    'BlastHeader.Xml',
    'BlisData.Xml',
    'SPNETData.Xml',
    'UIDefaults.Xml',
]

SPF_FILE

WindowsPath('../data/1505708.spf')

## Extract files from the SPF container

This cell:
- checks that the file exists,
- opens it as a ZIP-like container,
- extracts the target files if present,
- skips any missing files.


In [3]:
def ensure_spf_exists(spf_path: Path) -> None:
    if not spf_path.exists():
        raise FileNotFoundError(f'SPF file not found: {spf_path.resolve()}')


def safe_extract_member(zf: zipfile.ZipFile, member_name: str, out_dir: Path) -> Optional[Path]:
    """Extract a specific file by basename match, case-insensitive.

    Returns the extracted path if found, else None.
    """
    members = zf.namelist()
    member_lookup = {Path(m).name.lower(): m for m in members}
    key = member_name.lower()

    if key not in member_lookup:
        return None

    archive_member = member_lookup[key]
    extracted_path = Path(zf.extract(archive_member, path=out_dir))

    # Normalise location so the extracted file sits directly in out_dir with a clean name.
    final_path = out_dir / member_name
    final_path.parent.mkdir(parents=True, exist_ok=True)
    if extracted_path.resolve() != final_path.resolve():
        shutil.copy2(extracted_path, final_path)

    return final_path


def extract_spf_files(spf_path: Path, target_files: List[str], extract_root: Path) -> Dict[str, Path]:
    ensure_spf_exists(spf_path)
    extract_dir = extract_root / spf_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)

    extracted: Dict[str, Path] = {}

    if not zipfile.is_zipfile(spf_path):
        raise ValueError(
            f'{spf_path.name} is not recognised as a ZIP-compatible SPF container. '
            'If this SPF uses a different internal format, we will need a different extractor.'
        )

    with zipfile.ZipFile(spf_path, 'r') as zf:
        for target in target_files:
            extracted_path = safe_extract_member(zf, target, extract_dir)
            if extracted_path is not None and extracted_path.exists():
                extracted[target] = extracted_path

    return extracted


extracted_files = extract_spf_files(SPF_FILE, TARGET_FILES, EXTRACT_ROOT)
extracted_files

{'Header.Xml': WindowsPath('extracted_spf/1505708/Header.Xml'),
 'BlastHeader.Xml': WindowsPath('extracted_spf/1505708/BlastHeader.Xml'),
 'BlisData.Xml': WindowsPath('extracted_spf/1505708/BlisData.Xml'),
 'SPNETData.Xml': WindowsPath('extracted_spf/1505708/SPNETData.Xml'),
 'UIDefaults.Xml': WindowsPath('extracted_spf/1505708/UIDefaults.Xml')}

In [4]:
print('Extracted files:')
for name in TARGET_FILES:
    path = extracted_files.get(name)
    print(f'  {name:16s} -> {path if path else "missing"}')


Extracted files:
  Header.Xml       -> extracted_spf\1505708\Header.Xml
  BlastHeader.Xml  -> extracted_spf\1505708\BlastHeader.Xml
  BlisData.Xml     -> extracted_spf\1505708\BlisData.Xml
  SPNETData.Xml    -> extracted_spf\1505708\SPNETData.Xml
  UIDefaults.Xml   -> extracted_spf\1505708\UIDefaults.Xml


## Build the available file map

Only files that were actually found are included.


In [5]:
FILES = {
    'header': extracted_files.get('Header.Xml'),
    'blast_header': extracted_files.get('BlastHeader.Xml'),
    'blis': extracted_files.get('BlisData.Xml'),
    'spnet': extracted_files.get('SPNETData.Xml'),
    'ui_defaults': extracted_files.get('UIDefaults.Xml'),
}

FILES = {k: v for k, v in FILES.items() if v is not None and v.exists()}
FILES

{'header': WindowsPath('extracted_spf/1505708/Header.Xml'),
 'blast_header': WindowsPath('extracted_spf/1505708/BlastHeader.Xml'),
 'blis': WindowsPath('extracted_spf/1505708/BlisData.Xml'),
 'spnet': WindowsPath('extracted_spf/1505708/SPNETData.Xml'),
 'ui_defaults': WindowsPath('extracted_spf/1505708/UIDefaults.Xml')}

## Helper functions

In [6]:
def strip_ns(tag: str) -> str:
    if '}' in tag:
        return tag.split('}', 1)[1]
    return tag


def clean_col(name: str) -> str:
    name = re.sub(r'[^0-9a-zA-Z_]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    return name.lower()


def try_parse_value(value: Optional[str]) -> Any:
    if value is None:
        return None
    value = value.strip()
    if value == '':
        return None

    lower = value.lower()
    if lower == 'true':
        return True
    if lower == 'false':
        return False

    try:
        if re.fullmatch(r'[-+]?\d+', value):
            return int(value)
        if re.fullmatch(r'[-+]?\d*\.\d+(e[-+]?\d+)?', value.lower()) or re.fullmatch(r'[-+]?\d+e[-+]?\d+', value.lower()):
            return float(value)
    except Exception:
        pass

    return value


def xml_to_dict(elem: ET.Element, prefix: str = '') -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    children = list(elem)
    tag_name = strip_ns(elem.tag)

    if not children:
        key = clean_col(prefix[:-1] if prefix.endswith('_') else prefix or tag_name)
        out[key] = try_parse_value(elem.text)
        return out

    grouped: Dict[str, List[ET.Element]] = {}
    for child in children:
        grouped.setdefault(strip_ns(child.tag), []).append(child)

    for child_tag, elems in grouped.items():
        if len(elems) == 1:
            child = elems[0]
            child_children = list(child)
            child_prefix = f'{prefix}{child_tag}_'
            if child_children:
                out.update(xml_to_dict(child, prefix=child_prefix))
            else:
                out[clean_col(f'{prefix}{child_tag}')] = try_parse_value(child.text)
        else:
            for i, child in enumerate(elems):
                child_children = list(child)
                child_prefix = f'{prefix}{child_tag}_{i}_'
                if child_children:
                    out.update(xml_to_dict(child, prefix=child_prefix))
                else:
                    out[clean_col(f'{prefix}{child_tag}_{i}')] = try_parse_value(child.text)

    return out


def read_xml_root(path: Path) -> ET.Element:
    tree = ET.parse(path)
    return tree.getroot()


def first_child(root: ET.Element, tag_name: str) -> Optional[ET.Element]:
    for child in root:
        if strip_ns(child.tag) == tag_name:
            return child
    return None


def direct_children_by_name(parent: ET.Element, tag_name: str) -> List[ET.Element]:
    return [child for child in list(parent) if strip_ns(child.tag) == tag_name]


def records_to_df(records: Iterable[Dict[str, Any]]) -> pd.DataFrame:
    rows = list(records)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows)


def single_section_df(root: ET.Element, section_name: str) -> pd.DataFrame:
    section = first_child(root, section_name)
    if section is None:
        return pd.DataFrame()
    return pd.DataFrame([xml_to_dict(section, prefix='')])


def extract_repeating_section(root: ET.Element, parent_name: str, child_name: str) -> pd.DataFrame:
    parent = first_child(root, parent_name)
    if parent is None:
        return pd.DataFrame()
    rows = [xml_to_dict(child, prefix='') for child in direct_children_by_name(parent, child_name)]
    return records_to_df(rows)


def extract_nested_records(parent: Optional[ET.Element], path: List[str], repeated_child_name: str) -> pd.DataFrame:
    if parent is None:
        return pd.DataFrame()
    node = parent
    for step in path:
        node = first_child(node, step)
        if node is None:
            return pd.DataFrame()
    rows = [xml_to_dict(child, prefix='') for child in direct_children_by_name(node, repeated_child_name)]
    return records_to_df(rows)


## Load available XML roots

In [7]:
roots = {name: read_xml_root(path) for name, path in FILES.items()}

for name, root in roots.items():
    print(f'\n{name}: root={strip_ns(root.tag)}')
    print('top-level children:', [strip_ns(c.tag) for c in list(root)[:12]])



header: root=root
top-level children: ['FileHeader']

blast_header: root=root
top-level children: ['BlastHeader']

blis: root=BlisBlastData
top-level children: ['BlastIdentification', 'BlastDescription', 'BlastDomains', 'Holes', 'Horizons', 'TieTypes', 'TieTable', 'Leadins', 'BlastBounds', 'UsageDesign', 'UsageActual', 'Outcomes']

spnet: root=root
top-level children: ['SPBlast']

ui_defaults: root=root
top-level children: ['UIDefaultValues']


## Metadata tables

In [8]:
df_header = single_section_df(roots['header'], 'FileHeader') if 'header' in roots else pd.DataFrame()
df_blast_header = single_section_df(roots['blast_header'], 'BlastHeader') if 'blast_header' in roots else pd.DataFrame()

blis_root = roots.get('blis')
if blis_root is not None:
    blast_ident = first_child(blis_root, 'BlastIdentification')
    blast_desc = first_child(blis_root, 'BlastDescription')
    df_blast_identification = pd.DataFrame([xml_to_dict(blast_ident)]) if blast_ident is not None else pd.DataFrame()
    df_blast_description = pd.DataFrame([xml_to_dict(blast_desc)]) if blast_desc is not None else pd.DataFrame()
else:
    df_blast_identification = pd.DataFrame()
    df_blast_description = pd.DataFrame()

display(df_header)
display(df_blast_header)
display(df_blast_identification)
display(df_blast_description)


,fileversion,title,author,description,application,revision,thumbnail
0,1,1505708,DB Engineering,None,SHOTPlus6/6.20.5,35,/9j/4AAQSkZJRgABAQEAYABgAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/2wBDAQkJCQwLDBgNDRgyIRwhMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIy...


,blastguid,mine,location,comment,profilecomment,shotfirer,firingtime,blasttype,rocktype,orderno,vibtemplateid,maxdrilllength,surveyer,surveytime,boretracker,boretracktime,engineer,startloadingtime,finishloadingtime,customer,blastid,siteid,descriptor1,descriptor2,cosmossettings_blastcosmossettings_siteid,cosmossettings_blastcosmossettings_drillplanid,cosmossettings_blastcosmossettings_drillplanname,cosmossettings_blastcosmossettings_sitename,cosmossettings_blastcosmossettings_lastfullupdate,cosmossettings_blastcosmossettings_lastdeckhistoryupdate,cosmossettings_blastcosmossettings_isfired,cosmossettings_blastcosmossettings_blasttags,cosmossettings_blastcosmossettings_isdeleted,cosmossettings_blastcosmossettings_avmdomainid,cosmossettings_blastcosmossettings_avmdomainname,cosmossettings_blastcosmossettings_spatialtransform,cosmossettings_blastcosmossettings_lastdesignupdateid,cosmossettings_blastcosmossettings_cosmossite_cosmossite_id,cosmossettings_blastcosmossettings_cosmossite_cosmossite_name,cosmossettings_blastcosmossettings_cosmossite_cosmossite_type,cosmossettings_blastcosmossettings_cosmossite_cosmossite_timezone,cosmossettings_blastcosmossettings_cosmossite_cosmossite_countrycode,cosmossettings_blastcosmossettings_cosmossite_cosmossite_latitude,cosmossettings_blastcosmossettings_cosmossite_cosmossite_longitude,cosmossettings_blastcosmossettings_cosmossite_cosmossite_isdeleted,cosmossettings_blastcosmossettings_cosmossite_cosmossite_datemodified,cosmossettings_blastcosmossettings_cosmossite_cosmossite_datecreated,cosmossettings_blastcosmossettings_cosmosenv
0,5da31a37-42b7-4c72-bf21-cabdc5db1e36,Peñasquito,F7,None,None,None,None,None,None,None,00000000-0000-0000-0000-000000000000,5,None,None,None,None,None,None,None,None,1505708,None,None,None,88c75265-458d-4798-9702-da0f5d050c89,a25c29ee-9c8c-4840-85d0-0194d652208e,1505708,Newmont - Mina Penasquito,2024-12-19T12:04:12.5082006-06:00,0001-01-01T00:00:00-06:00,None,None,None,None,None,None,b2c6b228-1f3b-47a0-a90f-7e3b8d6b412b,88c75265-458d-4798-9702-da0f5d050c89,Newmont - Mina Penasquito,OpenCut,Central Standard Time (Mexico),MEX,24.640176,-101.671507,False,2025-02-03T21:01:56.1310738-06:00,2020-04-26T21:07:24.5355614-05:00,Production


,application,version,filename,spguid,author,description,planbearing,viewbearing,shotfirer,surveyor,boretracker,thumbnail,blasttrackid
0,SP5,6.20.5,1505708.spf,5da31a37-42b7-4c72-bf21-cabdc5db1e36,DB Engineering,None,42.709386,0,None,None,None,/9j/4AAQSkZJRgABAQEAYABgAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/2wBDAQkJCQwLDBgNDRgyIRwhMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIy...,1505708


,volumedesign,volumedesignmethod,powderfactordesign,energyfactordesign,isebs,volumeactualmethod,numredrills,redrilllength,numbackfills,backfilllength,numdecksouttol,decklengthvar,deckweightvar,issurveyed,islaidout,isdesigned,isloaded,ispartiallyloaded,isdipped,isboretraked,blastcentre_x,blastcentre_y,blastcentre_z,overlaptimewindow,backfillproductname,altbackfillproductname
0,143996.597473,ByHole,1.567598,0.096194,True,Manual,294,0,0,0,3,36,0,False,False,True,False,False,False,False,229401.55,2729037.55,1520.1,8,Stemming Backfill - World,Stemming Backfill - World


## Repeated sections from `BlisData.Xml`

In [9]:
df_domains = extract_repeating_section(blis_root, 'BlastDomains', 'BlastDomain') if blis_root is not None else pd.DataFrame()
df_holes = extract_repeating_section(blis_root, 'Holes', 'Hole') if blis_root is not None else pd.DataFrame()
df_horizons = extract_repeating_section(blis_root, 'Horizons', 'Horizon') if blis_root is not None else pd.DataFrame()
df_usage_design = extract_repeating_section(blis_root, 'UsageDesign', 'Resource') if blis_root is not None else pd.DataFrame()
df_usage_actual = extract_repeating_section(blis_root, 'UsageActual', 'Resource') if blis_root is not None else pd.DataFrame()
df_firing_times = extract_repeating_section(blis_root, 'FiringTimes', 'FiringTime') if blis_root is not None else pd.DataFrame()

for name, df in {
    'domains': df_domains,
    'holes': df_holes,
    'horizons': df_horizons,
    'usage_design': df_usage_design,
    'usage_actual': df_usage_actual,
    'firing_times': df_firing_times,
}.items():
    print(f'{name:12s} -> shape={df.shape}')

display(df_domains.head())
display(df_holes.head())
display(df_usage_design.head())
display(df_firing_times.head())


domains      -> shape=(24, 20)
holes        -> shape=(297, 88)
horizons     -> shape=(25, 2)
usage_design -> shape=(6, 7)
usage_actual -> shape=(0, 0)
firing_times -> shape=(291, 9)


,index,name,burden,spacing,subdrill,angle,diameter,benchheight,pattern,backfilltol,redrilltol,loadingtol,stemmingmaxtol,stemmingmintol,burdenmaxtol,burdenmintol,powderfactormaxtol,powderfactormintol,subdrillmaxtol,subdrillmintol
0,0,BUFFER 1,3.0,6.0,0.0,0,270,15,Staggered,0.0,-1.0,0.1,4.4,3.6,3.30,2.70,0.1101,0,0.5,0
1,1,BUFFER 2,8.0,6.0,0.0,0,270,15,Staggered,0.0,-1.0,0.1,5.4,4.6,8.70,7.30,0.1101,0,0.5,0
2,2,PROD,8.0,9.0,1.0,0,311,15,Staggered,0.0,-1.0,0.1,7.4,6.6,8.55,7.45,0.1101,0,1.5,0
3,3,PROD AGUA,8.0,9.0,1.0,0,311,15,Staggered,0.0,-1.0,0.1,7.9,7.1,8.70,7.30,0.1101,0,1.5,0
4,4,CONTORNOS 311,8.0,9.0,1.0,0,311,15,Staggered,0.0,-1.0,0.1,11.4,10.6,8.70,7.30,0.1101,0,1.5,0


,index,guid,holeid,domain,designlength,diameter,waterstate,designcoordinates_x,designcoordinates_y,designcoordinates_z,geocoordinates,designangle,designbearing,benchheight,subdrill,graderl,isdummy,isloaded,ispartiallyloaded,isboretraked,sequence,geocoordinates_latitude,geocoordinates_longitude,geocoordinates_hmsl,actualcoordinates_x,actualcoordinates_y,actualcoordinates_z,actualangle,actualbearing,comment,stemming,backfill,powderfactor,designloading_decks_holeindex,designloading_decks_deckindex,designloading_decks_explosive,designloading_decks_abbreviation,designloading_decks_horizon,designloading_decks_length,designloading_decks_density,designloading_decks_weight,designloading_decks_type,designloading_decks_sapcode,designloading_decks_rid,initsysloading,hasloading,hasactual,ebsflags,rulename,actualbackfill,actualcoalbackfill,loadinglength,firingtime,designloading_decks_0_holeindex,designloading_decks_0_deckindex,designloading_decks_0_explosive,designloading_decks_0_abbreviation,designloading_decks_0_horizon,designloading_decks_0_length,designloading_decks_0_density,designloading_decks_0_weight,designloading_decks_0_type,designloading_decks_0_sapcode,designloading_decks_0_rid,designloading_decks_1_holeindex,designloading_decks_1_deckindex,designloading_decks_1_explosive,designloading_decks_1_abbreviation,designloading_decks_1_horizon,designloading_decks_1_length,designloading_decks_1_density,designloading_decks_1_weight,designloading_decks_1_type,designloading_decks_1_sapcode,designloading_decks_1_rid,initsysloading_initiators_holeindex,initsysloading_initiators_initiatorindex,initsysloading_initiators_deckindex,initsysloading_initiators_product,initsysloading_initiators_depth,initsysloading_initiators_firingtime,initsysloading_initiators_primername,initsysloading_initiators_primerweight,initsysloading_initiators_primercount,initsysloading_initiators_isebs,initsysloading_initiators_rid,initsysloading_initiators_primerrid,initsysloading_initiators_drxrid
0,0,{e67265d2-9ce0-452b-b4f0-7432bee4ebb0},NaN,0,0.0,0,Unknown,229439.6,2729076.1,1534.8,NaN,0,0,0.0,0.0,0,True,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,{f0860d3e-2d75-4bf5-b5b7-8cbe4c70fe08},NaN,0,0.0,0,Unknown,229417.5,2729058.7,1534.7,NaN,0,0,0.0,0.0,0,True,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,{790945a4-ebe8-4852-b0dd-866b969eceae},NaN,0,0.0,0,Unknown,229441.5,2729032.2,1535.2,NaN,0,0,0.0,0.0,0,True,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,{8311f49d-9ba3-4ccb-8e62-ee665b6e884a},1.0,7,8.0,270,Unknown,229410.2,2729053.2,1520.9,NaN,0,0,6.8,1.2,0,False,False,False,False,NaN,24.652033,-101.673372,0.0,229410.2,2729053.2,1520.9,0.0,0.0,NaN,0.0,0.0,0.0,3.0,0.0,Air,Air,0.0,8.0,0.0,0.0,Empty,QAIR,3979ab73-32f7-450f-93ab-cb2ebb95783f,NaN,False,False,0.0,NaN,-1.0,-1.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,{98ff027f-9b97-4ae7-91d5-a33ff4e9c079},2.0,7,8.0,270,Unknown,229379.2,2729018.8,1520.9,NaN,0,0,6.8,1.2,0,False,False,False,False,NaN,24.651717,-101.673673,0.0,229379.2,2729018.8,1520.9,0.0,0.0,NaN,0.0,0.0,0.0,4.0,0.0,Air,Air,0.0,8.0,0.0,0.0,Empty,QAIR,3979ab73-32f7-450f-93ab-cb2ebb95783f,NaN,False,False,0.0,NaN,-1.0,-1.0,8.0,

,resourcetype,name,amount,unittype,rid,code,density
0,Stemming,Air,1.374133,Volume,3979ab73-32f7-450f-93ab-cb2ebb95783f,Air,NaN
1,Drill,"Drill, 270.0mm",4899.400000,Length,"DRLL,AUS,User defined,270.00",NaN,NaN
2,Initiator,"i-kon III Extreme - 20m, 0ms",291.000000,Count,"d6a41e40-8d6c-4423-aa3b-0077a0ab01ff,476324ad-facd-6a33-0df3-4453abd15abd",NaN,NaN
3,Primer,Pentex CD 450,291.000000,Count,"1c999970-51e6-4ff0-8e0e-b5717498fd8d,bdac964b-da35-4816-b060-7751ac776df1",NaN,NaN
4,Stemming,Stemming-Coal,91.036287,Volume,f5ddcca0-439e-41eb-be3e-de80cf007279,None,NaN


,holeindex,chargeindex,horizonindex,firingtime,chargeweight,devicemeandelay,devicestddev,chargetop,chargebottom
0,6,0,1,2462,425.981114,2462,0,10,16.2
1,7,0,1,2488,419.110451,2488,0,10,16.1
2,8,0,1,2531,439.722441,2531,0,10,16.4
3,9,0,1,2545,453.463767,2545,0,10,16.6
4,10,0,1,2383,762.643608,2383,0,5,16.1


## Selected sections from `SPNETData.Xml`

In [10]:
sp_root_outer = roots.get('spnet')
sp_root = first_child(sp_root_outer, 'SPBlast') if sp_root_outer is not None else None

df_expl_resources = extract_nested_records(sp_root, ['ResourceMgr', 'ResourceMgr', 'Resources'], 'ExplResource')
df_is_resources = extract_nested_records(sp_root, ['ResourceMgr', 'ResourceMgr', 'Resources'], 'ISResource')
df_air_resources = extract_nested_records(sp_root, ['ResourceMgr', 'ResourceMgr', 'Resources'], 'AirResource')

print('expl_resources:', df_expl_resources.shape)
print('is_resources:  ', df_is_resources.shape)
print('air_resources: ', df_air_resources.shape)

display(df_expl_resources.head())
display(df_is_resources.head())


expl_resources: (0, 0)
is_resources:   (0, 0)
air_resources:  (0, 0)


""


""


## Collect all parsed tables

In [11]:
tables = {
    'header': df_header,
    'blast_header': df_blast_header,
    'blast_identification': df_blast_identification,
    'blast_description': df_blast_description,
    'domains': df_domains,
    'holes': df_holes,
    'horizons': df_horizons,
    'usage_design': df_usage_design,
    'usage_actual': df_usage_actual,
    'firing_times': df_firing_times,
    'expl_resources': df_expl_resources,
    'is_resources': df_is_resources,
    'air_resources': df_air_resources,
}

for name, df in tables.items():
    print(f'{name:20s} {df.shape}')


header               (1, 7)
blast_header         (1, 48)
blast_identification (1, 13)
blast_description    (1, 26)
domains              (24, 20)
holes                (297, 88)
horizons             (25, 2)
usage_design         (6, 7)
usage_actual         (0, 0)
firing_times         (291, 9)
expl_resources       (0, 0)
is_resources         (0, 0)
air_resources        (0, 0)


## Optional export of parsed tables

In [ ]:
EXPORT_DIR = Path('parsed_spf_tables') / SPF_FILE.stem
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for name, df in tables.items():
    if not df.empty:
        df.to_csv(EXPORT_DIR / f'{name}.csv', index=False)

print(f'Exported CSVs to: {EXPORT_DIR.resolve()}')


## Notes

- If a file is not present inside the SPF, the notebook skips it.
- If the SPF is not ZIP-compatible, the extraction cell will tell you that explicitly.
- Once you confirm the exact internal file set used by your SPF exports, you can expand `TARGET_FILES` and add more extractors.
